# F1 · Definición del proyecto

**Qué hace este notebook:**
- Define el problema, las preguntas, los objetivos y el alcance del proyecto.
- Registra el entorno reproducible con el que se trabajó.

**Qué no hace este notebook:**
- No analiza datos ni compara modelos. Eso se hace en los notebooks F2 y F3.

## 1. Problema

**Contexto:** hoy se decide qué modelo de lenguaje es "mejor" mirando votos de personas que comparan dos respuestas anónimas, como en [_arena-human-preference-140k_](https://huggingface.co/datasets/lmarena-ai/arena-human-preference-140k) (Chiang et al., 2024). Si has usado ChatGPT, seguro te han pedido feedback sobre _qué respuesta prefieres_.

Se asume que ese voto refleja la calidad de la respuesta, pero puede haber otros factores que estén sesgando esta evaluación. Por ejemplo, una respuesta bien formateada, con títulos, listas y texto en negrita puede verse mejor y ser elegida más por su estética y por producir mayor estímulo de lectura, aunque no sea más correcta. En evaluadores automáticos ya se ha documentado un sesgo hacia respuestas más largas (Dubois et al., 2024). En votos humanos, en cambio, no sabemos si ocurre lo mismo con el formato. Un estudio sobre anotadores humanos y jueces basados en LLM encontró que las respuestas con errores factuales se valoraban más que las demasiado cortas o con errores gramaticales (Wu & Aji, 2025).

Eso significaría que el ranking premia el formato por sobre la calidad real.

**Con los datos de _arena-human-preference-140k_ se medirá si el formato y el largo de la respuesta están asociados con ganar.**

## 2. Pregunta de investigación

**Pregunta 1: Estilo vs. calidad:** ¿gana más seguido la respuesta más larga y con más formato?

**Restricción:** es una pregunta de **asociación**, no de causa.

## 3. Objetivos

**Objetivo general:** determinar si el formato (encabezados, listas y negritas) y el largo de las respuestas se asocian con que el modelo gane un voto, a partir de 135.634 votos de texto de [LMArena](https://huggingface.co/datasets/lmarena-ai/arena-human-preference-140k) entre el 17 de abril y el 24 de julio de 2025.

**Objetivos específicos** (F1 y F2 preparan el dato; el análisis de asociación es F3):

1. Describir la estructura del dataset y las distribuciones de formato, largo y resultado, identificando desbalances y valores atípicos (F2).
2. Preparar esas variables: aplanar `conv_metadata`, ajustar tipos, crear variables derivadas y guardar el resultado en `data/processed/` (F2).
3. Validar el dataset procesado con pruebas automatizadas antes de analizar P1 (F2).


## 4. Alcance, definición operativa y supuestos

**Definición operativa (Contenido vs. Formato):**
- **Contenido:** calidad semántica, corrección factual, coherencia argumentativa y pertinencia de la respuesta respecto al prompt.
- **Formato (estilo visual y estructura):** métricas cuantitativas de extensión y marcado Markdown en `conv_metadata`: conteo de encabezados (`h1`–`h6`), listas (ordenadas y no ordenadas), negritas (`**`, `__`) y longitud en tokens.

**Unidad de análisis y rol de las variables:**
- **Unidad de análisis:** voto humano individual por comparación ciega a pares (modelo A vs. modelo B) dentro de una sesión.
- **Variable dependiente:** resultado de preferencia (`winner`: `model_a`, `model_b`, `tie`, `both_bad`, y su versión simplificada `winner_binary`).
- **Variables independientes:** conteos y sumas de formato (`format_a`, `format_b`, `more_formatted`) y extensión (`sum_assistant_a/b_tokens`, `tokens_log_a/b`, `longer_response`).
- **Variables de control:** idioma (`language`), modelos comparados (`model_a`, `model_b`), código (`is_code`) y etiquetas de categoría (`category_tag`).

**Incluye:**

- Solo votos de la categoría de texto (135.634 votos, 17-abr al 24-jul de 2025, 53 modelos, 126 idiomas).
- Las variables de formato (encabezados, listas, negritas), el largo de las respuestas en tokens y el resultado del voto.
- "Ganar" significa que el voto eligió ese modelo. Los empates y los votos "ambos malos" se analizan aparte.

**No incluye:**

- El análisis de asociación entre formato, largo y resultado, que corresponde a Fase 3.
- Causas: solo asociaciones.
- Evaluar la calidad real de las respuestas. Medimos preferencias, no corrección.
- Tendencias de largo plazo: el periodo cubre unos tres meses.

**Supuestos y restricciones:**

- El voto refleja la preferencia de quien vota, no una medida objetiva de calidad.
- El dataset no tiene valores nulos. Sus limitaciones están en los empates, en los 126 idiomas, en 10 filas donde `model_a == model_b` y en 61 y 58 respuestas con cero tokens (ver `F2/F2_01_acquisition_exploration.ipynb`, sección 9).
- Los prompts tienen licencia CC-BY-4.0; las respuestas dependen de cada proveedor, por lo que no se redistribuyen textos y el dato crudo (1,6 GB) se descarga con `make data` y se verifica con `SHA-256` (`CHECKSUMS.txt`).

## 5. Entorno reproducible

**Qué hacemos:** registrar la versión de Python, la ruta del intérprete y el commit con el que se ejecutó el notebook.

**Por qué:** es la evidencia de reproducibilidad que pide la rúbrica. Si otra persona obtiene valores distintos, lo primero que se compara es este bloque.

In [1]:
from pathlib import Path
from arena.env import environment_info

info = environment_info()

resumen = {k: v for k, v in info.items() if k != "dependencies"}

resumen["interpreter"] = Path(
    *Path(info["interpreter"]).parts[-3:]
).as_posix()

resumen

{'python': '3.12.10',
 'system': 'Windows 11',
 'interpreter': '.venv/Scripts/python.exe',
 'virtual_env': True,
 'seed': 42,
 'git': {'branch': 'docs/report-review', 'commit': 'fb11abf', 'clean': 'True'}}

## 6. Código base del flujo

**Qué hacemos:** fijar la semilla y definir las rutas de datos crudos y procesados, que usarán todos los notebooks del proyecto.

**Por qué:** cualquier paso con aleatoriedad, como el muestreo de F2, se repite igual, y las rutas no dependen del lugar desde donde se ejecute el notebook.

In [2]:
# RAW_DIR es la carpeta donde `make data` deja el dataset descargado y verificado
from arena.io import RAW_DIR

SEED = 42  # semilla fija: cualquier muestreo se repite igual en otro equipo
PROCESSED_DIR = RAW_DIR.parent / "processed"

print("Semilla:", SEED)
print("Datos crudos:", RAW_DIR.relative_to(RAW_DIR.parent.parent))
print("Datos procesados:", PROCESSED_DIR.relative_to(RAW_DIR.parent.parent))

Semilla: 42
Datos crudos: data\raw
Datos procesados: data\processed


In [3]:
from arena.io import list_remote_files

archivos = list_remote_files()

print("Cantidad de archivos", len(archivos))
print("Primer archivo", archivos[0])

Cantidad de archivos 7
Primer archivo data/train-00000-of-00007.parquet


In [4]:
from huggingface_hub import hf_hub_download
from arena.io import REPO_ID, RAW_DIR

ruta_archivo = hf_hub_download(
    repo_id=REPO_ID,
    filename=archivos[0],
    repo_type="dataset",
    local_dir=RAW_DIR
)

print("Archivo descargado:", archivos[0])

Archivo descargado: data/train-00000-of-00007.parquet


In [5]:
import pandas as pd

muestra = pd.read_parquet(
    ruta_archivo,
    columns=["model_a", "model_b", "winner"]
).head(5)

display(muestra)

,model_a,model_b,winner
0,gemini-2.5-pro,claude-3-7-sonnet-20250219-thinking-32k,model_a
1,claude-3-5-haiku-20241022,claude-3-5-sonnet-20241022,tie
2,o3-mini,gemma-3n-e4b-it,both_bad
3,claude-sonnet-4-20250514-thinking-32k,gemini-2.5-pro,model_a
4,claude-3-5-sonnet-20241022,mistral-medium-2505,model_b


In [6]:
print("Tamaño de la muestra", muestra.shape)

print("Columnas", muestra.columns.tolist())

print("Tipos de datos:")
print(muestra.dtypes)

Tamaño de la muestra (5, 3)
Columnas ['model_a', 'model_b', 'winner']
Tipos de datos:
model_a    str
model_b    str
winner     str
dtype: object


In [7]:
columnas_necesarias = ["model_a", "model_b", "winner"]

print("¿Están las columnas necesarias?")
print(all(columna in muestra.columns for columna in columnas_necesarias))

print("Valores faltantes por columna:")
print(muestra.isna().sum())

¿Están las columnas necesarias?
True
Valores faltantes por columna:
model_a    0
model_b    0
winner     0
dtype: int64


In [8]:
resultados_validos = ["model_a", "model_b", "tie", "both_bad"]

print("¿Los resultados son válidos?")
print(muestra["winner"].isin(resultados_validos).all())

¿Los resultados son válidos?
True


### Revisión inicial de los datos

Se extrajo una muestra de 5 registros del dataset y se revisó los tipos de datos, que estuvieran las columnas que necesitábamos y que no hubiera valores faltantes. Como última validación, se buscó que todos los resultados de la muestra fuesen válidos.

Esta revisión se realizó únicamente sobre los cinco registros seleccionados, no sobre el dataset completo.

In [9]:
import pyarrow.parquet as pq

archivo = pq.ParquetFile(ruta_archivo)

cantidad_filas = archivo.metadata.num_rows
cantidad_columnas = len(archivo.schema_arrow.names)

print("Cantidad de filas:", cantidad_filas )
print("Cantidad de columnas:", cantidad_columnas )

print("¿Cumple con el mínimo de 2.000 filas?", cantidad_filas >= 2000)
print("¿Cumple con el mínimo de 12 columnas?", cantidad_columnas >= 12)

Cantidad de filas: 19377
Cantidad de columnas: 14
¿Cumple con el mínimo de 2.000 filas? True
¿Cumple con el mínimo de 12 columnas? True


### Revisión de requisitos del dataset

Al revisar uno de los archivos del dataset, se pudo comprobar que contiene 19.377 filas y 14 columnas, por lo que cumple con los mínimos solicitados de 2.000 filas y 12 columnas.

Cabe destacar que para esta revisión se utilizó solo un parquet de los 7 que compone el dataset completo.

## 7. Vinculación con el mapa conceptual

El mapa conceptual del grupo es el siguiente.

![Mapa conceptual del proyecto](mapa-conceptual-revisado.png)


## 8. Referencias

Chiang, W.-L., Zheng, L., Sheng, Y., Angelopoulos, A. N., Li, T., Li, D., Zhu, B., Zhang, H., Jordan, M., Gonzalez, J. E., & Stoica, I. (2024). *Chatbot Arena: An open platform for evaluating LLMs by human preference*. En *Proceedings of the 41st International Conference on Machine Learning* (Vol. 235, pp. 8359–8388). PMLR. https://proceedings.mlr.press/v235/chiang24b.html

Dubois, Y., Galambosi, B., Liang, P., & Hashimoto, T. B. (2024). *Length-controlled AlpacaEval: A simple way to debias automatic evaluators*. arXiv. https://arxiv.org/abs/2404.04475

LMArena. (2025). *arena-human-preference-140k* [Conjunto de datos]. Hugging Face. https://huggingface.co/datasets/lmarena-ai/arena-human-preference-140k

Wu, M., & Aji, A. F. (2025). *Style over substance: Evaluation biases for large language models*. En *Proceedings of the 31st International Conference on Computational Linguistics* (pp. 297–312). Association for Computational Linguistics. https://aclanthology.org/2025.coling-main.21